# Mini RAG: Retrieve Then Answer (No Hype)

**Purpose:** Show where a vector index fits in “RAG” — without turning this into an LLM ad.

> **RAG in one sentence:**  
> **Retrieve relevant context → answer grounded in that context.**

This notebook focuses on:
- grounding answers in retrieved passages
- showing citations explicitly
- demonstrating failure when grounding is skipped

We will:
1. Briefly define RAG
2. Retrieve top passages using embeddings + FAISS
3. Answer the question using retrieved context
   - **Option A (default):** no LLM, simple extractive summary
   - **Option B (optional):** LLM call *only if* user provides an API key
4. Print citations
5. Show what goes wrong if you skip retrieval


## 0) Setup

We’ll use:
- `sentence-transformers` for embeddings
- `faiss` for retrieval
- basic Python for extractive answering

The LLM step is **optional** and clearly isolated.


In [27]:
# Install dependencies if needed (safe to re-run)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    !uv add sentence-transformers

try:
    import faiss  # noqa: F401
except ImportError:
    !uv add faiss-cpu

In [28]:
import numpy as np
import pandas as pd
import faiss
import re

from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 160)

## 1) Build a small knowledge corpus

Think of these as:
- document chunks
- FAQ entries
- handbook sections

RAG only works if the answer *exists in the corpus*.


In [29]:
docs = [
    ("D01", "Banks assess credit risk before approving loans. They evaluate income stability, credit history, and existing debt."),
    ("D02", "High-interest debt should usually be paid down first because it accumulates interest faster over time."),
    ("D03", "An emergency fund helps cover unexpected expenses and typically contains three to six months of living costs."),
    ("D04", "A loyalty program can increase customer retention by rewarding repeat purchases."),
    ("D05", "Customer retention improves when support resolves issues quickly and effectively."),
    ("D06", "Refactoring code reduces technical debt and makes systems easier to maintain."),
    ("D07", "Profiling a program helps identify performance bottlenecks before optimization."),
    ("D08", "Monitoring official advisories is important when a typhoon is nearby."),
    ("D09", "Preparing emergency supplies such as water, food, and batteries helps during severe weather."),
    ("D10", "Breathing exercises can reduce stress in the short term by slowing the nervous system."),
]

df = pd.DataFrame(docs, columns=["doc_id", "text"])
df

,doc_id,text
0,D01,"Banks assess credit risk before approving loans. They evaluate income stability, credit history, and existing debt."
1,D02,High-interest debt should usually be paid down first because it accumulates interest faster over time.
2,D03,An emergency fund helps cover unexpected expenses and typically contains three to six months of living costs.
3,D04,A loyalty program can increase customer retention by rewarding repeat purchases.
4,D05,Customer retention improves when support resolves issues quickly and effectively.
5,D06,Refactoring code reduces technical debt and makes systems easier to maintain.
6,D07,Profiling a program helps identify performance bottlenecks before optimization.
7,D08,Monitoring official advisories is important when a typhoon is nearby.
8,D09,"Preparing emergency supplies such as water, food, and batteries helps during severe weather."
9,D10,Breathing exercises can reduce stress in the short term by slowing the nervous system.


## 2) Build embeddings + FAISS index

This is the **retrieval layer** of RAG.


In [30]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

texts = df["text"].tolist()
emb = model.encode(texts, normalize_embeddings=True).astype("float32")

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine-like with normalized embeddings
index.add(emb)

print("Docs:", len(df), "| Index size:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Docs: 10 | Index size: 10


## 3) Retrieve top passages

This step is the **grounding**.


In [31]:
def retrieve(query, k=3):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    rows = []
    for rank, (i, score) in enumerate(zip(idx[0], scores[0]), start=1):
        rows.append({
            "rank": rank,
            "doc_id": df.loc[i, "doc_id"],
            "score": float(score),
            "text": df.loc[i, "text"],
        })
    return pd.DataFrame(rows)

# Example retrieval
retrieve("How do banks decide whether to approve a loan?", k=3)

,rank,doc_id,score,text
0,1,D01,0.607012,"Banks assess credit risk before approving loans. They evaluate income stability, credit history, and existing debt."
1,2,D02,0.291840,High-interest debt should usually be paid down first because it accumulates interest faster over time.
2,3,D08,0.147846,Monitoring official advisories is important when a typhoon is nearby.


## 4) Answering step — Option A (No LLM)

We’ll create a **very simple extractive answer**:
- Select the most relevant sentence(s) from retrieved passages
- Stitch them together

This is intentionally basic to emphasize:
> Retrieval quality matters more than generation flair.


In [32]:
def extractive_answer(retrieved_df, max_sentences=2):
    sentences = []
    for text in retrieved_df["text"]:
        # naive sentence split
        for s in re.split(r"[.!?]", text):
            s = s.strip()
            if s:
                sentences.append(s)
    return ". ".join(sentences[:max_sentences]) + "."

def answer_without_llm(query, k=3):
    retrieved = retrieve(query, k=k)
    answer = extractive_answer(retrieved)
    return answer, retrieved

# Demo
q = "How do banks decide whether to approve a loan?"
ans, cites = answer_without_llm(q, k=3)
print("ANSWER:", ans)
print("CITATIONS:")
display(cites)

ANSWER: Banks assess credit risk before approving loans. They evaluate income stability, credit history, and existing debt.
CITATIONS:


,rank,doc_id,score,text
0,1,D01,0.607012,"Banks assess credit risk before approving loans. They evaluate income stability, credit history, and existing debt."
1,2,D02,0.291840,High-interest debt should usually be paid down first because it accumulates interest faster over time.
2,3,D08,0.147846,Monitoring official advisories is important when a typhoon is nearby.


### Why this works

- The answer is **fully grounded** in retrieved text
- No hallucination beyond what’s in the corpus
- Citations are explicit

This is often good enough for:
- internal tools
- decision support
- educational systems


## 5) Answering step — Option B (Optional LLM)

⚠️ **Optional and off by default**

If you *choose* to use an LLM:
- You pass the retrieved passages as context
- You instruct the model to answer *only* from that context
- You still print citations

This notebook does NOT require an API key.
A placeholder function is provided to show structure only.


In [33]:
def answer_with_llm_placeholder(query, retrieved_df):
    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not contained in the context, say "I don't know."

Question:
{query}

Context:
"""
    for i, row in retrieved_df.iterrows():
        prompt += f"[{row['doc_id']}] {row['text']}\n"
    
    return {
        "prompt_example": prompt,
        "note": "This is where an LLM call would happen if configured."
    }

# Example (no actual LLM call)
answer_with_llm_placeholder(q, cites)

{'prompt_example': '\nAnswer the question using ONLY the context below.\nIf the answer is not contained in the context, say "I don\'t know."\n\nQuestion:\nHow do banks decide whether to approve a loan?\n\nContext:\n[D01] Banks assess credit risk before approving loans. They evaluate income stability, credit history, and existing debt.\n[D02] High-interest debt should usually be paid down first because it accumulates interest faster over time.\n[D08] Monitoring official advisories is important when a typhoon is nearby.\n',
 'note': 'This is where an LLM call would happen if configured.'}

Let's use Ollama!

Download and install via this link: https://ollama.com/download

In [49]:
# Install dependencies if needed (safe to re-run)
try:
    import ollama  # noqa: F401
except ImportError:
    !uv add ollama
    !ollama pull "qwen3:0.6b"

In [50]:
from ollama import chat
from ollama import ChatResponse


def answer_with_llm_ollama(
    query: str,
    retrieved_df,
    model: str = "qwen3:0.6b",
    temperature: float = 0.0,
):
    # Build context block
    context = "\n".join(
        f"[{row['doc_id']}] {row['text']}"
        for _, row in retrieved_df.iterrows()
    )

    system = (
        "You are a careful assistant. "
        "Use ONLY the provided context. "
        "Paraphrasing is allowed, but you must NOT add new facts. "
        "If the context does not contain enough information to answer, reply exactly: I don't know."
    )

    user = (
        f"Question: {query}\n\n"
        f"Context:\n{context}\n\n"
        "Write a 1–2 sentence answer. "
        "If you use a claim, cite the supporting context IDs like [D01]."
    )

    resp = chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        options={"temperature": temperature},
    )

    return {
        "answer": resp.message.content,
        "used_model": model,
        "prompt_system": system,
        "prompt_user": user,
    }



In [51]:
q = "How do banks decide whether to approve a loan?"

result = answer_with_llm_ollama(q, cites)

print("ANSWER:")
print(result["answer"])

print("\n--- PROMPT SENT TO MODEL ---\n")
print(result["used_model"])
print(result["prompt_system"])
print(result["prompt_user"])



ANSWER:
Banks assess credit risk by evaluating income stability, credit history, and existing debt before approving loans. They prioritize these factors to determine approval, as mentioned in [D01].

--- PROMPT SENT TO MODEL ---

qwen3:0.6b
You are a careful assistant. Use ONLY the provided context. Paraphrasing is allowed, but you must NOT add new facts. If the context does not contain enough information to answer, reply exactly: I don't know.
Question: How do banks decide whether to approve a loan?

Context:
[D01] Banks assess credit risk before approving loans. They evaluate income stability, credit history, and existing debt.
[D02] High-interest debt should usually be paid down first because it accumulates interest faster over time.
[D08] Monitoring official advisories is important when a typhoon is nearby.

Write a 1–2 sentence answer. If you use a claim, cite the supporting context IDs like [D01].


## 6) Citations (good habit)

Always show:
- which passages were used
- where the answer came from

This:
- builds trust
- makes errors debuggable
- discourages hallucination


## 7) Failure case: skip grounding → hallucination risk

What happens if we **answer without retrieval**?

The system may:
- invent details
- give generic advice
- sound confident but be unsupported

We won’t even simulate an LLM here — the lesson is conceptual.


### Takeaway

- **Vector search is not the product** — it’s a *supporting component*
- RAG works when:
  - the corpus contains the answer
  - retrieval is good
  - generation is constrained by context

> No retrieval → no grounding → no trust.


## Outputs checklist

- ✅ grounded answer
- ✅ retrieved passages printed as citations
- ✅ optional (clearly marked) LLM step
- ✅ clear failure explanation when grounding is skipped
